# Aula 17 · Lab: adicionando tarefa If/else e automatizando seu job (resolução comentada)

## 1. Parâmetro
**Job parameters → Edit** → `limite_risco_pct` = `10`.

## 2. If/else `risco_alto`
**+ Add task → If/else condition**
- **Task name:** `risco_alto` · **Depends on:** `gold`
- **Condition:** `{{tasks.gold.values.taxa_inadimplencia_pct}}` **>** `{{job.parameters.limite_risco_pct}}`

> Os dois lados aceitam referências dinâmicas: de um lado o recado da tarefa, do outro o parâmetro do job. O limite muda sem mexer em código.

## 3. Ramo verdadeiro
**+ Add task → Notebook** → `alerta_risco` · Path `Projeto/06_alerta_risco` · **Depends on:** `risco_alto (true)` · Parameters `job_run_id` = `{{job.run_id}}`.

O ramo falso não precisa de tarefa: se o risco está normal, o fluxo só segue.

## 4. `registrar_execucao` continua rodando sempre
Edite `registrar_execucao` → **Depends on:** `gold`, `alerta_risco` e `registrar_alerta_qualidade` · **Run if:** *All done*.

> Por que não depender de `risco_alto`? Uma tarefa que depende de um If/else precisa escolher um ramo (*true* ou *false*). Dependendo de `alerta_risco`, o log é gravado **depois** da decisão, qualquer que seja o resultado.

## 5. Execução automática
**Schedules & Triggers → Add trigger → File arrival** → `/Volumes/growdev/bronze/landing/loans/`.

## 6. Testes
| Teste | Como | Resultado esperado |
|---|---|---|
| Ramo falso | **Run now** (limite 10) | `alerta_risco` = Excluded |
| Ramo verdadeiro | **Run now with different parameters** → `limite_risco_pct` = `5` | `alerta_risco` = Succeeded |
| Disparo automático | Rode `Projeto/utilitarios/simular_chegada_lote` | Nova execução com trigger *File arrival* |

## O trecho do job como código
```yaml
      parameters:
        - name: limite_risco_pct
          default: "10"
      trigger:
        file_arrival:
          url: /Volumes/growdev/bronze/landing/loans/
      tasks:
        - task_key: risco_alto
          depends_on:
            - task_key: gold
          condition_task:
            left: "{{tasks.gold.values.taxa_inadimplencia_pct}}"
            op: GREATER_THAN
            right: "{{job.parameters.limite_risco_pct}}"
        - task_key: alerta_risco
          depends_on:
            - task_key: risco_alto
              outcome: "true"
          notebook_task:
            notebook_path: .../Projeto/06_alerta_risco
            base_parameters:
              job_run_id: "{{job.run_id}}"
        - task_key: registrar_execucao
          depends_on:
            - task_key: gold
            - task_key: alerta_risco
            - task_key: registrar_alerta_qualidade
          run_if: ALL_DONE
```

In [0]:
%sql
SELECT job_run_id, state, customer_segment, taxa_pct, taxa_carteira_pct, registrado_em
FROM growdev.gold.alertas_risco
ORDER BY registrado_em DESC, taxa_pct DESC
LIMIT 20